<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Instalar las librerías necesarias
# !pip install google-generativeai matplotlib requests pandas

import time
import io
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import google.generativeai as genai
from google.colab import userdata

# ==========================================
# 1. CONFIGURACIÓN DE CREDENCIALES
# ==========================================
# Clave de API de Google AI Studio extraída de tu imagen
GEMINI_API_KEY = "AQ.Ab8RN6KycZOV613RC9wVfrdGwbC_TETgy7shEfk6hUcHHGIZA"
genai.configure(api_key=GEMINI_API_KEY)

# Token de GitHub (puedes guardarlo en Colab Secrets como 'GITHUB_TOKEN' o dejarlo opcional)
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    GITHUB_TOKEN = "" # Opcional: Reemplaza con tu token de GitHub si deseas sincronizar

# ==========================================
# 2. MOTOR DE CÁLKULO DE COSTOS (REGLAS EXACTAS)
# ==========================================
def calcular_costos(params):
    t_inicio = time.time()

    material_ajustado = params['consumo_estandar'] * (1 + (params['desperdicio'] / 100))
    costo_mp_unitario = material_ajustado * params['precio_material']
    costo_mod_unitario = params['horas_mod'] * params['tarifa_mod']
    costo_variable_unitario = costo_mp_unitario + costo_mod_unitario + params['cif_variable']
    costo_total = (costo_variable_unitario * params['volumen']) + params['costos_fijos']
    costo_unitario = costo_total / params['volumen'] if params['volumen'] > 0 else 0
    margen_contribucion_unitario = params['precio_venta'] - costo_variable_unitario

    if margen_contribucion_unitario <= 0:
        punto_equilibrio = float('inf')
        advertencia = "⚠️ ADVERTENCIA: Margen <= 0. Punto de equilibrio no válido o inalcanzable."
    else:
        punto_equilibrio = params['costos_fijos'] / margen_contribucion_unitario
        advertencia = "✅ OK (Trazable y Válido)"

    utilidad = (margen_contribucion_unitario * params['volumen']) - params['costos_fijos']

    t_fin = time.time()
    tiempo_simulacion_ms = (t_fin - t_inicio) * 1000

    return {
        'material_ajustado': material_ajustado,
        'costo_mp_unitario': costo_mp_unitario,
        'costo_mod_unitario': costo_mod_unitario,
        'costo_variable_unitario': costo_variable_unitario,
        'costo_total': costo_total,
        'costo_unitario': costo_unitario,
        'margen_contribucion_unitario': margen_contribucion_unitario,
        'punto_equilibrio': punto_equilibrio,
        'utilidad': utilidad,
        'advertencia': advertencia,
        'tiempo_simulacion_ms': tiempo_simulacion_ms
    }

# Parámetros base
base_params = {
    'material': 'Acero Inoxidable',
    'consumo_estandar': 1.0,
    'precio_material': 50.0,
    'desperdicio': 4.0,
    'horas_mod': 2.5,
    'tarifa_mod': 15.0,
    'cif_variable': 10.0,
    'costos_fijos': 5000.0,
    'volumen': 1000,
    'precio_venta': 120.0
}

# ==========================================
# 3. EJECUCIÓN DE PRUEBAS OBLIGATORIAS
# ==========================================
escenarios_prueba = {
    "Base": base_params.copy(),
    "Materia Prima +12%": {**base_params, 'precio_material': base_params['precio_material'] * 1.12},
    "Desperdicio 9%": {**base_params, 'desperdicio': 9.0},
    "Extremo (Margen <= 0)": {**base_params, 'precio_venta': 65.0, 'cif_variable': 35.0}
}

resultados_tabla = []
datos_completos_json = {}
nombres_escenarios = []
utilidades = []
costos_unitarios = []

utilidad_base = None

for nombre, params in escenarios_prueba.items():
    res = calcular_costos(params)
    nombres_escenarios.append(nombre)
    utilidades.append(res['utilidad'])
    costos_unitarios.append(res['costo_unitario'])
    datos_completos_json[nombre] = {**params, **res}

    if "Base" in nombre:
        utilidad_base = res['utilidad']
        var_utilidad = 0.0
    else:
        var_utilidad = res['utilidad'] - utilidad_base

    resultados_tabla.append({
        'Escenario': nombre,
        'Costo Unitario': f"${res['costo_unitario']:.2f}",
        'Margen Unitario': f"${res['margen_contribucion_unitario']:.2f}",
        'Utilidad Total': f"${res['utilidad']:.2f}",
        'Var. vs Base': f"${var_utilidad:.2f}",
        'Validación': res['advertencia']
    })

df_pruebas = pd.DataFrame(resultados_tabla)
print("="*90)
print("REPORTE DE PRUEBAS OBLIGATORIAS Y ESCENARIOS")
print("="*90)
display(df_pruebas)

# ==========================================
# 4. GENERACIÓN DE GRÁFICA COMPARATIVA (SVG)
# ==========================================
plt.style.use('default')
fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:blue'
ax1.set_ylabel('Utilidad Total ($)', color=color, fontweight='bold')
bars = ax1.bar(nombres_escenarios, utilidades, color=color, alpha=0.8, width=0.4, label='Utilidad')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_title('Simulación Industrial: Utilidad vs Costo Unitario por Escenario', fontsize=12, fontweight='bold', pad=15)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Costo Unitario ($)', color=color, fontweight='bold')
ax2.plot(nombres_escenarios, costos_unitarios, color=color, marker='o', linewidth=2.5, label='Costo Unitario')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()

# Guardar en buffer SVG
svg_buffer = io.BytesIO()
plt.savefig(svg_buffer, format='svg', bbox_inches='tight')
svg_buffer.seek(0)
svg_content = svg_buffer.getvalue().decode('utf-8')
plt.close()
print("\n✅ Gráfica SVG generada correctamente.")

# ==========================================
# 5. ANÁLISIS GERENCIAL CON GOOGLE GEMINI
# ==========================================
res_base = calcular_costos(base_params)

contexto_numerico = f"""
RESULTADOS CALCULADOS POR EL MOTOR (ESCENARIO BASE):
- Material analizado: {base_params['material']}
- Volumen de producción/ventas: {base_params['volumen']} unidades
- Precio de venta unitario: ${base_params['precio_venta']:.2f}
- Material ajustado: {res_base['material_ajustado']:.4f}
- Costo MP unitario: ${res_base['costo_mp_unitario']:.2f}
- Costo MOD unitario: ${res_base['costo_mod_unitario']:.2f}
- Costo variable unitario: ${res_base['costo_variable_unitario']:.2f}
- Costo total: ${res_base['costo_total']:.2f}
- Costo unitario total: ${res_base['costo_unitario']:.2f}
- Margen de contribución unitario: ${res_base['margen_contribucion_unitario']:.2f}
- Punto de equilibrio: {res_base['punto_equilibrio']:.2f} unidades
- Utilidad operativa: ${res_base['utilidad']:.2f}
"""

prompt_ia = f"""
Actúa como analista de costos industriales.
Recibirás resultados YA CALCULADOS por un simulador:
{contexto_numerico}

Tu tarea es:
1) Explicar qué indican estos resultados respecto a la sensibilidad de la utilidad, el margen y el punto de equilibrio.
2) Ordenar las variables de mayor a menor impacto cuantitativo según los datos provistos.
3) Proponer hasta 3 acciones concretas y cuantificables para probar en el simulador.
4) Señalar los supuestos y riesgos asociados.

Reglas estrictas que DEBES cumplir:
- No inventes costos, precios ni volúmenes que no estén en los datos provistos.
- No sustituyas las fórmulas matemáticas programadas en el motor.
- Toda propuesta de acción debe indicar explícitamente qué variable modificar y en qué dirección.
- Distingue claramente el resultado calculado de la recomendación analítica.
- No afirmes que una alternativa es óptima sin una comparación numérica directa.
- Haz evidentes los supuestos sobre los cuales descansan tus recomendaciones.

Formato de salida requerido:
- Diagnóstico:
- Variables críticas:
- Acciones a simular:
- Riesgos:
"""

print("\n" + "="*90)
print("DIAGNÓSTICO GERENCIAL (GENERADO POR GOOGLE GEMINI)")
print("="*90)

try:
    # Usando el modelo Gemini configurado con tu clave
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt_ia)
    print(response.text)
except Exception as e:
    print(f"Error al generar análisis con Gemini: {e}")

# ==========================================
# 6. ENVÍO AUTOMÁTICO A GITHUB GIST
# ==========================================
if not GITHUB_TOKEN:
    print("\n⚠️ Nota: No se configuró el GITHUB_TOKEN. Los resultados y la gráfica se procesaron localmente en Colab.")
else:
    print("\n🔄 Sincronizando resultados y gráficas con GitHub Gist...")
    gist_payload = {
        "description": "Simulación de Costos Industriales - Reporte, Escenarios y Gráfica",
        "public": False,
        "files": {
            "resultados_costos.json": {
                "content": json.dumps(datos_completos_json, indent=2, default=str)
            },
            "comparativa_escenarios.svg": {
                "content": svg_content
            }
        }
    }

    headers = {
        'Authorization': f'token {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github.v3+json',
        'Content-Type': 'application/json'
    }

    response = requests.post('https://api.github.com/gists', headers=headers, data=json.dumps(gist_payload))

    if response.status_code == 201:
        gist_data = response.json()
        print("¡Enviado a GitHub con éxito! 🚀")
        print(f"🔗 Enlace al Gist privado: {gist_data['html_url']}")
    else:
        print(f"❌ Error al conectar con GitHub API: {response.status_code} - {response.text}")

REPORTE DE PRUEBAS OBLIGATORIAS Y ESCENARIOS


,Escenario,Costo Unitario,Margen Unitario,Utilidad Total,Var. vs Base,Validación
0,Base,$104.50,$20.50,$15500.00,$0.00,✅ OK (Trazable y Válido)
1,Materia Prima +12%,$110.74,$14.26,$9260.00,$-6240.00,✅ OK (Trazable y Válido)
2,Desperdicio 9%,$107.00,$18.00,$13000.00,$-2500.00,✅ OK (Trazable y Válido)
3,Extremo (Margen <= 0),$129.50,$-59.50,$-64500.00,$-80000.00,⚠️ ADVERTENCIA: Margen <= 0. Punto de equilibr...



✅ Gráfica SVG generada correctamente.

DIAGNÓSTICO GERENCIAL (GENERADO POR GOOGLE GEMINI)


Error al generar análisis con Gemini: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.

⚠️ Nota: No se configuró el GITHUB_TOKEN. Los resultados y la gráfica se procesaron localmente en Colab.
